In [0]:
%sql

CREATE OR REPLACE TABLE dev.sales.customers_cdc_target (
    customer_id INT,
    customer_name STRING,
    email STRING,
    status STRING,
    updated_at TIMESTAMP
)
USING DELTA;

INSERT INTO dev.sales.customers_cdc_target
(customer_id, customer_name, email, status, updated_at)
VALUES
    (1, 'Anna Kowalska', 'anna@example.com', 'ACTIVE', current_timestamp()),
    (2, 'Bartek Nowak', 'bartek@example.com', 'ACTIVE', current_timestamp()),
    (3, 'Celina Ziel', 'celina@example.com', 'ACTIVE', current_timestamp());

CREATE OR REPLACE TABLE dev.sales.customers_cdc_events (
    op STRING,
    customer_id INT,
    customer_name STRING,
    email STRING,
    status STRING,
    event_ts TIMESTAMP
)
USING DELTA;

INSERT INTO dev.sales.customers_cdc_events
(op, customer_id, customer_name, email, status, event_ts)
VALUES
    ('U', 1, 'Anna Nowak', 'anna.nowak@example.com', 'ACTIVE', current_timestamp()),
    ('D', 2, NULL, NULL, NULL, current_timestamp()),
    ('I', 4, 'Daniel Test', 'daniel@example.com', 'ACTIVE', current_timestamp());

In [0]:
%sql
SELECT *
FROM dev.sales.customers_cdc_target;

In [0]:
%sql
SELECT *
FROM dev.sales.customers_cdc_events;

In [0]:
%sql
MERGE INTO dev.sales.customers_cdc_target AS t
USING dev.sales.customers_cdc_events AS s
ON t.customer_id = s.customer_id

WHEN MATCHED AND s.op = 'D' THEN
  DELETE

WHEN MATCHED AND s.op = 'U' THEN
  UPDATE SET
    t.customer_name = s.customer_name,
    t.email = s.email,
    t.status = s.status,
    t.updated_at = s.event_ts

WHEN NOT MATCHED AND s.op = 'I' THEN
  INSERT (
    customer_id,
    customer_name,
    email,
    status,
    updated_at
  )
  VALUES (
    s.customer_id,
    s.customer_name,
    s.email,
    s.status,
    s.event_ts
  );

In [0]:
%sql
SELECT *
FROM dev.sales.customers_cdc_target
ORDER BY customer_id;

In [0]:
%sql
DESCRIBE HISTORY dev.sales.customers_cdc_target;